In [1]:
!pip install onnx
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 9.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms

import onnx

import copy

In [3]:
ds_train = datasets.MNIST('data', train=True, download=True, transform=transforms.ToTensor())
ds_test = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())

100%|██████████| 9.91M/9.91M [00:00<00:00, 60.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.75MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.05MB/s]


In [4]:
class ALU(nn.Module):
  def __init__(self):
    super(ALU, self).__init__()

  def forward(self, x):
    return x * ((1/torch.pi) * torch.arctan(x) + 0.5)

In [5]:
class ScaledGeLU(nn.Module):
  def __init__(self, alpha):
    super(ScaledGeLU, self).__init__()
    self.alpha = alpha

  def forward(self, x):
    return self.alpha * F.gelu((1/self.alpha) * x)

In [6]:
def convert_scaled_to_standard(sequential_net):
    """
    Converts a Sequential model using ScaledGeLU to use standard nn.GELU
    by absorbing the alpha scaling factor into adjacent Linear/Conv layers.
    """
    with torch.no_grad():
        # We work on a copy or the original; here we modify in-place
        layers = sequential_net

        i = 0
        while i < len(layers):
            if isinstance(layers[i], ScaledGeLU):
                alpha = layers[i].alpha

                # --- Step 1: Adjust the PRECEDING layer ---
                # We need to find the first parameter-heavy layer before the activation
                prev_idx = i - 1
                if prev_idx >= 0:
                    prev_layer = layers[prev_idx]
                    if hasattr(prev_layer, 'weight'):
                        prev_layer.weight.data /= alpha
                        if prev_layer.bias is not None:
                            prev_layer.bias.data /= alpha
                    else:
                        print(f"Warning: Layer {prev_idx} has no weights to scale down.")

                # --- Step 2: Adjust the SUCCEEDING layer ---
                # We need to find the first parameter-heavy layer after the activation
                next_idx = i + 1
                if next_idx < len(layers):
                    next_layer = layers[next_idx]
                    if hasattr(next_layer, 'weight'):
                        # Weights are multiplied by alpha because the input to
                        # this layer is now alpha-times smaller than before.
                        next_layer.weight.data *= alpha
                        # NOTE: The bias of the NEXT layer remains UNCHANGED.
                        # Because: next_layer(alpha * GELU) = W*(alpha*GELU) + bias
                    else:
                        print(f"Warning: Layer {next_idx} has no weights to scale up.")

                # --- Step 3: Replace the activation ---
                layers[i] = nn.GELU()

            i += 1
    return sequential_net

# Setup

## Model Construction

In [7]:
def make_model(width, depth, input_size, num_classes, actfun='relu', scale_actfun=1.):
  if scale_actfun == 1.:
    if actfun == 'relu':
      f = nn.ReLU
    elif actfun == 'gelu':
      f = nn.GELU
    elif actfun == 'alu':
      f = ALU
    elif actfun == 'sigmoid':
      f = torch.nn.Sigmoid
    else:
      raise ValueError(f'Unknown activation function: {actfun}')
  else:
    if actfun == 'gelu':
      f = ScaledGeLU
    else:
      raise ValueError(f'Unknown activation function: {actfun}')

  model = nn.Sequential()
  model.add_module('flatten', nn.Flatten())
  model.add_module('input', nn.Linear(input_size, width))
  if scale_actfun != 1.:
    model.add_module(f'{actfun}0', f(scale_actfun))
  else:
    model.add_module(f'{actfun}0', f())

  for i in range(1,depth):
    layer = nn.Linear(width, width)
    model.add_module(f'fc{i}', layer)
    if scale_actfun != 1.:
      model.add_module(f'{actfun}{i}', f(scale_actfun))
    else:
      model.add_module(f'{actfun}{i}', f())

  layer_out = nn.Linear(width, num_classes)
  model.add_module(f'fc{depth}', layer_out)

  return model

## Training Function

Function to ensure *only weights* and not biases are penalized with L1 term.

Large biases are commonly excluded from this regularization. And that's also what we prefer in our case!

If the algo is allowed to produce biases that shift far away from zero we are in the more linear regions of our activation functions.

In [8]:
def compute_l1_norm(model):
  l1 = 0
  for name, param in model.named_parameters():
    # Check if the parameter name contains 'weight' and does NOT contain 'bias'
    # The 'weight' check is often redundant if you skip 'bias', but good practice.
    # The key is to check for 'bias' in the name and skip it.
    if 'bias' not in name:
      l1 += torch.sum(torch.abs(param))

  return l1

In [9]:
def train(model, train_loader, optimizer, l1_weight=0):
  model.train()
  train_loss = 0
  correct = 0
  for batch_idx, (X, y) in enumerate(train_loader):
    optimizer.zero_grad()
    y_pred = model(X)
    loss = F.cross_entropy(y_pred, y)

    if l1_weight > 0:
      l1_reg = compute_l1_norm(model)
      loss += l1_weight * l1_reg

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = y_pred.argmax(dim=1, keepdim=True)
    correct += pred.eq(y.view_as(pred)).sum().item()

  train_loss = train_loss / len(train_loader.dataset)
  accuracy = 100. * correct / len(train_loader.dataset)

  print(f'Train set: Average loss: {train_loss:.4f}, Accuracy: {correct}/{len(train_loader.dataset)} ({accuracy:.0f}%)')

  return train_loss, accuracy

## Test Function

In [10]:
def test(model, test_loader):
  model.eval()
  test_loss = 0
  correct = 0
  with torch.no_grad():
    for X, y in test_loader:
      y_pred = model(X)

      loss = F.cross_entropy(y_pred, y)
      test_loss += loss.item()

      pred = y_pred.argmax(dim=1, keepdim=True)
      correct += pred.eq(y.view_as(pred)).sum().item()

    test_loss = test_loss / len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)

    print(f'Test set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.0f}%)')

    return test_loss, accuracy

In [11]:
train_loader = torch.utils.data.DataLoader(ds_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(ds_test, batch_size=64, shuffle=True)

## Complete Training Loop

In [12]:
def train_model(width, depth, input_size, num_classes, actfun='relu', scale_actfun=1., l1_weight=0, epochs=20, init_model=None):
  model = make_model(width, depth, input_size, num_classes, actfun=actfun, scale_actfun=scale_actfun)

  if init_model:
    model.load_state_dict(init_model)

  optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

  best_loss = float('inf')
  best_model = None
  for epoch in range(1, epochs+1):
    print(f'Epoch {epoch}')
    train_loss, train_acc = train(model, train_loader, optimizer, l1_weight=l1_weight)
    test_loss, test_acc = test(model, test_loader)

    if test_loss < best_loss:
      print("-- new best model")
      best_loss = test_loss
      best_model = copy.deepcopy(model.state_dict())

    l1_sum = compute_l1_norm(model)
    print("L1: ", l1_sum.item())

    print()


  model.load_state_dict(best_model)
  l1_sum = compute_l1_norm(model)
  print("L1: ", l1_sum.item())

  return model, best_model

# Model Training

## GeLU w/o L1-Loss

In [ ]:
model, best_model = train_model(256, 4, 784, 10, actfun='gelu', l1_weight=0, epochs=20)

Epoch 1
Train set: Average loss: 0.0046, Accuracy: 54517/60000 (91%)
Test set: Average loss: 0.0024, Accuracy: 9540/10000 (95%)
-- new best model
L1:  14168.8125

Epoch 2
Train set: Average loss: 0.0017, Accuracy: 57976/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9663/10000 (97%)
-- new best model
L1:  15891.666015625

Epoch 3
Train set: Average loss: 0.0012, Accuracy: 58607/60000 (98%)
Test set: Average loss: 0.0014, Accuracy: 9755/10000 (98%)
-- new best model
L1:  17469.591796875

Epoch 4
Train set: Average loss: 0.0009, Accuracy: 58978/60000 (98%)
Test set: Average loss: 0.0011, Accuracy: 9793/10000 (98%)
-- new best model
L1:  18850.705078125

Epoch 5
Train set: Average loss: 0.0007, Accuracy: 59145/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9813/10000 (98%)
-- new best model
L1:  20172.837890625

Epoch 6
Train set: Average loss: 0.0006, Accuracy: 59316/60000 (99%)
Test set: Average loss: 0.0014, Accuracy: 9788/10000 (98%)
L1:  21537.888671875

Epoch 7
Train s

In [ ]:
model_6, best_model_6 = train_model(256, 6, 784, 10, actfun='gelu', l1_weight=0, epochs=20)

Epoch 1
Train set: Average loss: 0.0055, Accuracy: 53335/60000 (89%)
Test set: Average loss: 0.0027, Accuracy: 9494/10000 (95%)
-- new best model
L1:  18794.4375

Epoch 2
Train set: Average loss: 0.0019, Accuracy: 57859/60000 (96%)
Test set: Average loss: 0.0018, Accuracy: 9669/10000 (97%)
-- new best model
L1:  20554.830078125

Epoch 3
Train set: Average loss: 0.0013, Accuracy: 58539/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9721/10000 (97%)
-- new best model
L1:  22143.2890625

Epoch 4
Train set: Average loss: 0.0010, Accuracy: 58901/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9715/10000 (97%)
L1:  23534.26953125

Epoch 5
Train set: Average loss: 0.0009, Accuracy: 59042/60000 (98%)
Test set: Average loss: 0.0015, Accuracy: 9759/10000 (98%)
-- new best model
L1:  24986.208984375

Epoch 6
Train set: Average loss: 0.0007, Accuracy: 59232/60000 (99%)
Test set: Average loss: 0.0015, Accuracy: 9745/10000 (97%)
L1:  26380.412109375

Epoch 7
Train set: Average loss: 0.0

## GeLU 2e-5 Regularization Strength

Same weight as in ReLU stability paper

In [ ]:
model_2e5, best_model_2e5 = train_model(256, 4, 784, 10, actfun='gelu', l1_weight=2e-5, epochs=20)

Epoch 1
Train set: Average loss: 0.0075, Accuracy: 54063/60000 (90%)
Test set: Average loss: 0.0020, Accuracy: 9585/10000 (96%)
-- new best model
L1:  7478.919921875

Epoch 2
Train set: Average loss: 0.0041, Accuracy: 57730/60000 (96%)
Test set: Average loss: 0.0017, Accuracy: 9685/10000 (97%)
-- new best model
L1:  6219.86083984375

Epoch 3
Train set: Average loss: 0.0032, Accuracy: 58362/60000 (97%)
Test set: Average loss: 0.0013, Accuracy: 9744/10000 (97%)
-- new best model
L1:  5404.06005859375

Epoch 4
Train set: Average loss: 0.0027, Accuracy: 58691/60000 (98%)
Test set: Average loss: 0.0012, Accuracy: 9747/10000 (97%)
-- new best model
L1:  4784.43896484375

Epoch 5
Train set: Average loss: 0.0024, Accuracy: 58909/60000 (98%)
Test set: Average loss: 0.0012, Accuracy: 9752/10000 (98%)
L1:  4378.25341796875

Epoch 6
Train set: Average loss: 0.0021, Accuracy: 59018/60000 (98%)
Test set: Average loss: 0.0013, Accuracy: 9759/10000 (98%)
L1:  4243.9833984375

Epoch 7
Train set: Averag

For GeLU already the `e-5` is too much, so we start with `5e-6` and go to the larger regularizer from there.

In [ ]:
model_6_2e5, best_model_6_2e5 = train_model(256, 6, 784, 10, actfun='gelu', l1_weight=2e-5, epochs=20, init_model=best_model_6)

Epoch 1
Train set: Average loss: 0.0045, Accuracy: 59312/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9802/10000 (98%)
-- new best model
L1:  9660.306640625

Epoch 2
Train set: Average loss: 0.0032, Accuracy: 59378/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9798/10000 (98%)
L1:  7756.07861328125

Epoch 3
Train set: Average loss: 0.0028, Accuracy: 59396/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9807/10000 (98%)
L1:  6820.96923828125

Epoch 4
Train set: Average loss: 0.0025, Accuracy: 59408/60000 (99%)
Test set: Average loss: 0.0014, Accuracy: 9792/10000 (98%)
L1:  6238.68994140625

Epoch 5
Train set: Average loss: 0.0023, Accuracy: 59459/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9805/10000 (98%)
L1:  5822.041015625

Epoch 6
Train set: Average loss: 0.0022, Accuracy: 59489/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9814/10000 (98%)
-- new best model
L1:  5378.47705078125

Epoch 7
Train set: Average loss: 0.0020, Accuracy: 59515/60000

## GeLU with 1e-4 Regularization Strength

In [ ]:
model_1e4, best_model_1e4 = train_model(256, 4, 784, 10, actfun='gelu', l1_weight=1e-4, epochs=20, init_model=best_model_2e5)

Epoch 1
Train set: Average loss: 0.0044, Accuracy: 59336/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9784/10000 (98%)
-- new best model
L1:  2178.353271484375

Epoch 2
Train set: Average loss: 0.0039, Accuracy: 59154/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9774/10000 (98%)
L1:  1929.872802734375

Epoch 3
Train set: Average loss: 0.0036, Accuracy: 59141/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9789/10000 (98%)
-- new best model
L1:  1811.9921875

Epoch 4
Train set: Average loss: 0.0034, Accuracy: 59130/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9766/10000 (98%)
L1:  1696.650146484375

Epoch 5
Train set: Average loss: 0.0033, Accuracy: 59118/60000 (99%)
Test set: Average loss: 0.0014, Accuracy: 9745/10000 (97%)
L1:  1612.0396728515625

Epoch 6
Train set: Average loss: 0.0031, Accuracy: 59121/60000 (99%)
Test set: Average loss: 0.0010, Accuracy: 9799/10000 (98%)
-- new best model
L1:  1516.1588134765625

Epoch 7
Train set: Average loss: 0.00

The large model doesn't learn at all when we directly start with the $L_1$ loss at $1e-4$, so we start with the $2e-5$ weight first and then train the resulting network with the $1e-4$ weight.

In [ ]:
model_6_1e4, best_model_6_1e4 = train_model(256, 6, 784, 10, actfun='gelu', l1_weight=1e-4, epochs=20, init_model=best_model_6_2e5)

Epoch 1
Train set: Average loss: 0.0050, Accuracy: 59471/60000 (99%)
Test set: Average loss: 0.0015, Accuracy: 9737/10000 (97%)
-- new best model
L1:  2572.360107421875

Epoch 2
Train set: Average loss: 0.0044, Accuracy: 59308/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9771/10000 (98%)
-- new best model
L1:  2264.515625

Epoch 3
Train set: Average loss: 0.0040, Accuracy: 59296/60000 (99%)
Test set: Average loss: 0.0014, Accuracy: 9747/10000 (97%)
L1:  2118.501953125

Epoch 4
Train set: Average loss: 0.0038, Accuracy: 59236/60000 (99%)
Test set: Average loss: 0.0009, Accuracy: 9829/10000 (98%)
-- new best model
L1:  1978.6456298828125

Epoch 5
Train set: Average loss: 0.0036, Accuracy: 59258/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9760/10000 (98%)
L1:  1914.6937255859375

Epoch 6
Train set: Average loss: 0.0035, Accuracy: 59211/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9779/10000 (98%)
L1:  1825.8878173828125

Epoch 7
Train set: Average loss: 0.0034,

# ONNX Export

In [ ]:
model.load_state_dict(best_model)
model_6.load_state_dict(best_model_6)

model_2e5.load_state_dict(best_model_2e5)
model_1e4.load_state_dict(best_model_1e4)

model_6_2e5.load_state_dict(best_model_6_2e5)
model_6_1e4.load_state_dict(best_model_6_1e4)

<All keys matched successfully>

In [ ]:
dummy_input = torch.randn(1, 1, 28, 28)

models = [model, model_6, model_2e5, model_1e4, model_6_2e5, model_6_1e4]
onnx_files = ['mnist_gelu_256x4.onnx', 'mnist_gelu_256x6.onnx', 'mnist_gelu_256x4_2e5.onnx', 'mnist_gelu_256x4_1e4.onnx',
              'mnist_gelu_256x6_2e5.onnx', 'mnist_gelu_256x6_1e4.onnx']

#models = [model_1e4]
#onnx_files = ['mnist_gelu_256x4_1e4.onnx']

for model, filename in zip(models, onnx_files):
    torch.onnx.export(model, dummy_input, filename, export_params=True, do_constant_folding=True, input_names=['input'], output_names=['output'], opset_version=20)

/tmp/ipython-input-2832491924.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(model, dummy_input, filename, export_params=True, do_constant_folding=True, input_names=['input'], output_names=['output'], opset_version=20)


# ALU

In [ ]:
model, best_model = train_model(256, 4, 784, 10, actfun='alu', l1_weight=0, epochs=20)

Epoch 1
Train set: Average loss: 0.0048, Accuracy: 54271/60000 (90%)
Test set: Average loss: 0.0022, Accuracy: 9571/10000 (96%)

Epoch 2
Train set: Average loss: 0.0019, Accuracy: 57754/60000 (96%)
Test set: Average loss: 0.0018, Accuracy: 9640/10000 (96%)

Epoch 3
Train set: Average loss: 0.0013, Accuracy: 58413/60000 (97%)
Test set: Average loss: 0.0015, Accuracy: 9709/10000 (97%)

Epoch 4
Train set: Average loss: 0.0010, Accuracy: 58734/60000 (98%)
Test set: Average loss: 0.0014, Accuracy: 9724/10000 (97%)

Epoch 5
Train set: Average loss: 0.0009, Accuracy: 58962/60000 (98%)
Test set: Average loss: 0.0012, Accuracy: 9794/10000 (98%)

Epoch 6
Train set: Average loss: 0.0007, Accuracy: 59160/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9745/10000 (97%)

Epoch 7
Train set: Average loss: 0.0006, Accuracy: 59280/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9797/10000 (98%)

Epoch 8
Train set: Average loss: 0.0005, Accuracy: 59404/60000 (99%)
Test set: Average loss: 0.00

In [ ]:
model_2e5, best_model_2e5 = train_model(256, 4, 784, 10, actfun='alu', l1_weight=2e-5, epochs=20)

Epoch 1
Train set: Average loss: 0.0078, Accuracy: 53520/60000 (89%)
Test set: Average loss: 0.0026, Accuracy: 9452/10000 (95%)

Epoch 2
Train set: Average loss: 0.0043, Accuracy: 57438/60000 (96%)
Test set: Average loss: 0.0018, Accuracy: 9649/10000 (96%)

Epoch 3
Train set: Average loss: 0.0034, Accuracy: 58085/60000 (97%)
Test set: Average loss: 0.0019, Accuracy: 9640/10000 (96%)

Epoch 4
Train set: Average loss: 0.0029, Accuracy: 58411/60000 (97%)
Test set: Average loss: 0.0014, Accuracy: 9715/10000 (97%)

Epoch 5
Train set: Average loss: 0.0025, Accuracy: 58687/60000 (98%)
Test set: Average loss: 0.0013, Accuracy: 9736/10000 (97%)

Epoch 6
Train set: Average loss: 0.0022, Accuracy: 58864/60000 (98%)
Test set: Average loss: 0.0013, Accuracy: 9745/10000 (97%)

Epoch 7
Train set: Average loss: 0.0021, Accuracy: 58962/60000 (98%)
Test set: Average loss: 0.0011, Accuracy: 9796/10000 (98%)

Epoch 8
Train set: Average loss: 0.0019, Accuracy: 59055/60000 (98%)
Test set: Average loss: 0.00

In [ ]:
model_1e4, best_model_1e4 = train_model(256, 4, 784, 10, actfun='alu', l1_weight=1e-4, epochs=20, init_model=best_model_2e5)

Epoch 1
Train set: Average loss: 0.0043, Accuracy: 59394/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9759/10000 (98%)

Epoch 2
Train set: Average loss: 0.0038, Accuracy: 59312/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9779/10000 (98%)

Epoch 3
Train set: Average loss: 0.0037, Accuracy: 59169/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9779/10000 (98%)

Epoch 4
Train set: Average loss: 0.0035, Accuracy: 59209/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9785/10000 (98%)

Epoch 5
Train set: Average loss: 0.0034, Accuracy: 59183/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9765/10000 (98%)

Epoch 6
Train set: Average loss: 0.0033, Accuracy: 59123/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9767/10000 (98%)

Epoch 7
Train set: Average loss: 0.0032, Accuracy: 59174/60000 (99%)
Test set: Average loss: 0.0011, Accuracy: 9787/10000 (98%)

Epoch 8
Train set: Average loss: 0.0032, Accuracy: 59118/60000 (99%)
Test set: Average loss: 0.00

In [ ]:
dummy_input = torch.randn(1, 1, 28, 28)

models = [model, model_2e5, model_1e4]
onnx_files = ['mnist_alu_256x4.onnx', 'mnist_alu_256x4_2e5.onnx', 'mnist_alu_256x4_1e4.onnx']

for model, filename in zip(models, onnx_files):
    torch.onnx.export(model, dummy_input, filename, export_params=True, do_constant_folding=True, input_names=['input'], output_names=['output'], opset_version=20)

/tmp/ipython-input-1985050957.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(model, dummy_input, filename, export_params=True, do_constant_folding=True, input_names=['input'], output_names=['output'], opset_version=20)


# Sigmoid

## w/o L1 Loss

In [ ]:
model, best_model = train_model(256, 4, 784, 10, actfun='sigmoid', l1_weight=0, epochs=20)

Epoch 1
Train set: Average loss: 0.0107, Accuracy: 46323/60000 (77%)
Test set: Average loss: 0.0046, Accuracy: 9148/10000 (91%)
-- new best model
L1:  26514.23828125

Epoch 2
Train set: Average loss: 0.0034, Accuracy: 56219/60000 (94%)
Test set: Average loss: 0.0029, Accuracy: 9466/10000 (95%)
-- new best model
L1:  29925.8671875

Epoch 3
Train set: Average loss: 0.0023, Accuracy: 57422/60000 (96%)
Test set: Average loss: 0.0022, Accuracy: 9581/10000 (96%)
-- new best model
L1:  32342.708984375

Epoch 4
Train set: Average loss: 0.0018, Accuracy: 58003/60000 (97%)
Test set: Average loss: 0.0022, Accuracy: 9598/10000 (96%)
-- new best model
L1:  34425.73828125

Epoch 5
Train set: Average loss: 0.0014, Accuracy: 58382/60000 (97%)
Test set: Average loss: 0.0017, Accuracy: 9691/10000 (97%)
-- new best model
L1:  36480.1171875

Epoch 6
Train set: Average loss: 0.0012, Accuracy: 58701/60000 (98%)
Test set: Average loss: 0.0019, Accuracy: 9658/10000 (97%)
L1:  38275.41015625

Epoch 7
Train set

## 2e-5 L1 Loss

We start with the best model from above, if we retrain from scratch with `2e-5`, the sigmoid model will not converge.

In [ ]:
model_2e5, best_model_2e5 = train_model(256, 4, 784, 10, actfun='sigmoid', l1_weight=2e-5, epochs=20, init_model=best_model)

Epoch 1
Train set: Average loss: 0.0089, Accuracy: 59126/60000 (99%)
Test set: Average loss: 0.0015, Accuracy: 9718/10000 (97%)
-- new best model
L1:  19415.83984375

Epoch 2
Train set: Average loss: 0.0063, Accuracy: 58873/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9678/10000 (97%)
L1:  15450.01171875

Epoch 3
Train set: Average loss: 0.0054, Accuracy: 58853/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9675/10000 (97%)
L1:  13295.5107421875

Epoch 4
Train set: Average loss: 0.0049, Accuracy: 58854/60000 (98%)
Test set: Average loss: 0.0017, Accuracy: 9656/10000 (97%)
L1:  12043.3017578125

Epoch 5
Train set: Average loss: 0.0046, Accuracy: 58848/60000 (98%)
Test set: Average loss: 0.0015, Accuracy: 9710/10000 (97%)
L1:  11128.6552734375

Epoch 6
Train set: Average loss: 0.0043, Accuracy: 58867/60000 (98%)
Test set: Average loss: 0.0014, Accuracy: 9726/10000 (97%)
-- new best model
L1:  10519.2392578125

Epoch 7
Train set: Average loss: 0.0041, Accuracy: 58913/60000

## 1e-4 L1 Loss

We start with the model from above as initialization again.

In [ ]:
model_1e4, best_model_1e4 = train_model(256, 4, 784, 10, actfun='sigmoid', l1_weight=1e-4, epochs=20, init_model=best_model_2e5)

Epoch 1
Train set: Average loss: 0.0101, Accuracy: 59025/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9700/10000 (97%)
-- new best model
L1:  5209.14013671875

Epoch 2
Train set: Average loss: 0.0088, Accuracy: 58694/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9714/10000 (97%)
-- new best model
L1:  4571.64111328125

Epoch 3
Train set: Average loss: 0.0082, Accuracy: 58503/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9716/10000 (97%)
L1:  4200.921875

Epoch 4
Train set: Average loss: 0.0078, Accuracy: 58450/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9703/10000 (97%)
L1:  3960.480712890625

Epoch 5
Train set: Average loss: 0.0074, Accuracy: 58475/60000 (97%)
Test set: Average loss: 0.0017, Accuracy: 9679/10000 (97%)
L1:  3757.2568359375

Epoch 6
Train set: Average loss: 0.0071, Accuracy: 58492/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9690/10000 (97%)
L1:  3586.818359375

Epoch 7
Train set: Average loss: 0.0069, Accuracy: 58434/60000 (9

In [ ]:
dummy_input = torch.randn(1, 1, 28, 28)

models = [model, model_2e5, model_1e4]
onnx_files = ['mnist_sigmoid_256x4.onnx', 'mnist_sigmoid_256x4_2e5.onnx', 'mnist_sigmoid_256x4_1e4.onnx']

for model_exp, filename in zip(models, onnx_files):
    model_exp.to('cpu')
    model_exp.eval()
    torch.onnx.export(model_exp,
                      dummy_input,
                      filename,
                      export_params=True,
                      do_constant_folding=True,
                      input_names=['input'],
                      output_names=['output'],
                      # dynamic_axes=dynamic_axes,
                      external_data=False) # Keep external_data=False to embed weights

W0203 16:52:45.315000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0203 16:52:45.317000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0203 16:52:45.320000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0203 16:52:45.322000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


W0203 16:52:46.394000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0203 16:52:46.396000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0203 16:52:46.398000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0203 16:52:46.402000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


W0203 16:52:47.484000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0203 16:52:47.486000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0203 16:52:47.488000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0203 16:52:47.492000 512 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


# Scaled GeLU

If we scale GeLU, s.t. it does not look similar to ReLU when zooming out, approximation performance should be better.

In [ ]:
model, best_model = train_model(256, 4, 784, 10, actfun='gelu', scale_actfun=5., l1_weight=0, epochs=20)

Epoch 1
Train set: Average loss: 0.0060, Accuracy: 52788/60000 (88%)
Test set: Average loss: 0.0031, Accuracy: 9369/10000 (94%)
-- new best model
L1:  14867.3310546875

Epoch 2
Train set: Average loss: 0.0028, Accuracy: 56670/60000 (94%)
Test set: Average loss: 0.0023, Accuracy: 9566/10000 (96%)
-- new best model
L1:  16999.205078125

Epoch 3
Train set: Average loss: 0.0021, Accuracy: 57525/60000 (96%)
Test set: Average loss: 0.0020, Accuracy: 9605/10000 (96%)
-- new best model
L1:  18648.255859375

Epoch 4
Train set: Average loss: 0.0017, Accuracy: 57973/60000 (97%)
Test set: Average loss: 0.0022, Accuracy: 9580/10000 (96%)
L1:  20166.73828125

Epoch 5
Train set: Average loss: 0.0015, Accuracy: 58224/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9704/10000 (97%)
-- new best model
L1:  21600.25

Epoch 6
Train set: Average loss: 0.0012, Accuracy: 58578/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9733/10000 (97%)
-- new best model
L1:  22930.802734375

Epoch 7
Train set

In [ ]:
model_2e5, best_model_2e5 = train_model(256, 4, 784, 10, actfun='gelu', scale_actfun=5., l1_weight=2e-5, epochs=20, init_model=best_model)

Epoch 1
Train set: Average loss: 0.0056, Accuracy: 59028/60000 (98%)
Test set: Average loss: 0.0014, Accuracy: 9730/10000 (97%)
-- new best model
L1:  11561.4541015625

Epoch 2
Train set: Average loss: 0.0040, Accuracy: 58963/60000 (98%)
Test set: Average loss: 0.0014, Accuracy: 9748/10000 (97%)
L1:  8851.140625

Epoch 3
Train set: Average loss: 0.0033, Accuracy: 58989/60000 (98%)
Test set: Average loss: 0.0012, Accuracy: 9782/10000 (98%)
-- new best model
L1:  7460.01220703125

Epoch 4
Train set: Average loss: 0.0030, Accuracy: 59018/60000 (98%)
Test set: Average loss: 0.0016, Accuracy: 9715/10000 (97%)
L1:  6676.91259765625

Epoch 5
Train set: Average loss: 0.0027, Accuracy: 59090/60000 (98%)
Test set: Average loss: 0.0013, Accuracy: 9774/10000 (98%)
L1:  6098.59521484375

Epoch 6
Train set: Average loss: 0.0025, Accuracy: 59142/60000 (99%)
Test set: Average loss: 0.0012, Accuracy: 9766/10000 (98%)
L1:  5675.8798828125

Epoch 7
Train set: Average loss: 0.0023, Accuracy: 59236/60000 (

In [ ]:
model_1e4, best_model_1e4 = train_model(256, 4, 784, 10, actfun='gelu', scale_actfun=5., l1_weight=1e-4, epochs=20, init_model=best_model)

Epoch 1
Train set: Average loss: 0.0131, Accuracy: 58319/60000 (97%)
Test set: Average loss: 0.0019, Accuracy: 9625/10000 (96%)
-- new best model
L1:  3894.53125

Epoch 2
Train set: Average loss: 0.0067, Accuracy: 58172/60000 (97%)
Test set: Average loss: 0.0022, Accuracy: 9568/10000 (96%)
L1:  2871.06396484375

Epoch 3
Train set: Average loss: 0.0056, Accuracy: 58311/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9677/10000 (97%)
-- new best model
L1:  2456.642333984375

Epoch 4
Train set: Average loss: 0.0051, Accuracy: 58341/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9690/10000 (97%)
L1:  2215.382568359375

Epoch 5
Train set: Average loss: 0.0047, Accuracy: 58440/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9681/10000 (97%)
L1:  2060.607421875

Epoch 6
Train set: Average loss: 0.0044, Accuracy: 58481/60000 (97%)
Test set: Average loss: 0.0017, Accuracy: 9657/10000 (97%)
L1:  1970.5032958984375

Epoch 7
Train set: Average loss: 0.0043, Accuracy: 58520/60000

In [ ]:
model

Sequential(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (input): Linear(in_features=784, out_features=256, bias=True)
  (gelu0): ScaledGeLU()
  (fc1): Linear(in_features=256, out_features=256, bias=True)
  (gelu1): ScaledGeLU()
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (gelu2): ScaledGeLU()
  (fc3): Linear(in_features=256, out_features=256, bias=True)
  (gelu3): ScaledGeLU()
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)

In [ ]:
convert_scaled_to_standard(model)

Sequential(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (input): Linear(in_features=784, out_features=256, bias=True)
  (gelu0): GELU(approximate='none')
  (fc1): Linear(in_features=256, out_features=256, bias=True)
  (gelu1): GELU(approximate='none')
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (gelu2): GELU(approximate='none')
  (fc3): Linear(in_features=256, out_features=256, bias=True)
  (gelu3): GELU(approximate='none')
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)

In [ ]:
dummy_input = torch.randn(1, 1, 28, 28)

models = [convert_scaled_to_standard(m) for m in [model, model_2e5, model_1e4]]
onnx_files = ['mnist_scaled_gelu_256x4.onnx', 'mnist_scaled_gelu_256x4_2e5.onnx', 'mnist_scaled_gelu_256x4_1e4.onnx']

for model_exp, filename in zip(models, onnx_files):
    model_exp.to('cpu')
    model_exp.eval()
    torch.onnx.export(model_exp,
                      dummy_input,
                      filename,
                      export_params=True,
                      do_constant_folding=True,
                      input_names=['input'],
                      output_names=['output'],
                      # dynamic_axes=dynamic_axes,
                      external_data=False) # Keep external_data=False to embed weights

W0205 14:26:10.709000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 14:26:10.711000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 14:26:10.713000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 14:26:10.715000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


W0205 14:26:13.022000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 14:26:13.023000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 14:26:13.025000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 14:26:13.027000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


W0205 14:26:14.100000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 14:26:14.101000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 14:26:14.104000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 14:26:14.106000 625 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


# Standard ReLU

In [13]:
model, best_model = train_model(256, 4, 784, 10, actfun='relu', l1_weight=0, epochs=20)

Epoch 1
Train set: Average loss: 0.0044, Accuracy: 54833/60000 (91%)
Test set: Average loss: 0.0020, Accuracy: 9615/10000 (96%)
-- new best model
L1:  13491.5615234375

Epoch 2
Train set: Average loss: 0.0017, Accuracy: 58078/60000 (97%)
Test set: Average loss: 0.0016, Accuracy: 9695/10000 (97%)
-- new best model
L1:  15161.912109375

Epoch 3
Train set: Average loss: 0.0012, Accuracy: 58558/60000 (98%)
Test set: Average loss: 0.0015, Accuracy: 9698/10000 (97%)
-- new best model
L1:  16710.17578125

Epoch 4
Train set: Average loss: 0.0009, Accuracy: 58959/60000 (98%)
Test set: Average loss: 0.0012, Accuracy: 9785/10000 (98%)
-- new best model
L1:  18006.650390625

Epoch 5
Train set: Average loss: 0.0007, Accuracy: 59135/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9771/10000 (98%)
L1:  19235.123046875

Epoch 6
Train set: Average loss: 0.0006, Accuracy: 59237/60000 (99%)
Test set: Average loss: 0.0013, Accuracy: 9796/10000 (98%)
L1:  20527.0625

Epoch 7
Train set: Average loss: 

In [14]:
dummy_input = torch.randn(1, 1, 28, 28)

models = [model]
onnx_files = ['mnist_relu_256x4.onnx']

for model_exp, filename in zip(models, onnx_files):
    model_exp.to('cpu')
    model_exp.eval()
    torch.onnx.export(model_exp,
                      dummy_input,
                      filename,
                      export_params=True,
                      do_constant_folding=True,
                      input_names=['input'],
                      output_names=['output'],
                      # dynamic_axes=dynamic_axes,
                      external_data=False) # Keep external_data=False to embed weights

[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
